<a href="https://colab.research.google.com/github/mlkanter/ee_kaart/blob/main/laanema_mets_analyys.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Läänemaa metsaandmete analüüs

Laadib alla avalikud metsaregistri ja looduskaitse andmed ning ühendab need üheks tabeliks Google Sheets'i jaoks.

**Väljund:**
- `laanema_mets.csv` — üks rida eraldis, kõik info koos
- `laanema_yoy.csv` — raiemaht aasta ja raieliigi kaupa

**Käivita lahtrid järjest ülalt alla. Iga lahter prindib kokkuvõtte kui õnnestub.**

In [1]:
# ── Lahter 1: Paigalda teegid (ainult esimesel käivitamisel ~30 sek) ──────────
!pip install geopandas requests pandas -q

import geopandas as gpd
import pandas as pd
import requests
import io

# Andmeallikad
WFS_MR    = "https://gsavalik.envir.ee/geoserver/metsaregister/ows"
WFS_EELIS = "https://gsavalik.envir.ee/geoserver/eelis/ows"

# Läänemaa piiritlev kast (lon_min, lat_min, lon_max, lat_max)
BBOX = "23.1,58.8,24.4,59.4,EPSG:4326"

# Raieliikide nimetused (too_kood -> eestikeelne)
TOO_NIMED = {
    "LR": "Lageraie",
    "HR": "Harvendusraie",
    "SR": "Sanitaarraie",
    "TR": "Turberaie",
    "VR": "Valikraie",
}

# Kaitsealatüüpide nimetused
KAITSEALA_NIMED = {
    "KMKA": "Maastikukaitseala",
    "KLKA": "Looduskaitseala",
    "KRP":  "Rahvuspark",
    "VK":   "Veekaitsevöönd",
    "PS":   "Püsielupaik",
    "KP":   "Kaitsepark",
    "VP":   "Veepark",
}

print("✓ Seadistus valmis")

✓ Seadistus valmis


In [3]:
# ── Lahter 2: Andmete laadimise abifunktsioon ─────────────────────────────────
def fetch_wfs(base_url, typename, bbox=None, cql=None, max_features=10000):
    """Laadib WFS kihi GeoDataFrame'ina. Tagastab None kui päring ebaõnnestub."""
    params = {
        "service": "WFS",
        "version": "2.0.0",
        "request": "GetFeature",
        "typeName": typename,
        "outputFormat": "application/json",
        "count": max_features,
    }
    if bbox: params["bbox"] = bbox
    if cql:  params["CQL_FILTER"] = cql

    print(f"  Laadin: {typename}...", end=" ")
    r = requests.get(base_url, params=params, timeout=120)
    if not r.ok:
        print(f"VIGA {r.status_code}")
        return None
    gdf = gpd.read_file(io.StringIO(r.text))
    print(f"{len(gdf)} rida")
    return gdf

print("✓ Abifunktsioon valmis")

✓ Abifunktsioon valmis


In [4]:
# ── Lahter 3: Lae metsaeraldised (põhitabel) ──────────────────────────────────
# Iga eraldis = üks metsa kvartalieraldis koos puistu infoga
eraldis = fetch_wfs(WFS_MR, "metsaregister:eraldis", bbox=BBOX)

# Vali analüüsiks vajalikud veerud
ERALDIS_VEERUD = [
    "katastri_nr",     # katastriüksuse number (liitmisvõti)
    "eraldise_nr",     # eraldise number kinnistul
    "pindala",         # pindala hektarites
    "peapuuliik_kood", # peapuuliik (MA=mänd, KU=kuusk, KS=kask, ...)
    "keskm_vanus",     # keskmine vanus aastates
    "omandivorm_kood", # omandivorm (R=riigi, M=munitsipaal, E=eraomand)
    "kasvukoht_kood",  # kasvukohatüüp
    "arengukl_kood",   # arenguklassi kood
    "tagavara_1_ha",   # puidu tagavara m³/ha
    "taius_1",         # puistu täius %
    "korgus",          # keskmine kõrgus
    "juurdekasv",      # aastane juurdekasv m³/ha
    "mkood",           # omavalitsuse kood
    "geometry",
]
eraldis = eraldis[[c for c in ERALDIS_VEERUD if c in eraldis.columns]]

print(f"\n✓ Eraldised laetud: {len(eraldis)} eraldist")
print(f"  Kogu pindala: {eraldis['pindala'].sum():.0f} ha")
print(f"  Peapuuliigid:\n{eraldis['peapuuliik_kood'].value_counts().head(8).to_string()}")

  Laadin: metsaregister:eraldis... 10000 rida

✓ Eraldised laetud: 10000 eraldist
  Kogu pindala: 13744 ha
  Peapuuliigid:
peapuuliik_kood
MA    3823
KS    3206
KU    1627
LM     564
HB     499
LV     114
SA      81
TA      67


In [5]:
# ── Lahter 4: Lae metsateatised (aktiivsed + arhiiv) ─────────────────────────
# teatis = kehtivad load; teatis_arhiiv = ajaloolised (YoY analüüsiks)
teatis        = fetch_wfs(WFS_MR, "metsaregister:teatis",        bbox=BBOX)
teatis_arhiiv = fetch_wfs(WFS_MR, "metsaregister:teatis_arhiiv", bbox=BBOX)

# Ühenda mõlemad
teatis_all = pd.concat(
    [teatis[[c for c in teatis.columns if c != "geometry"]],
     teatis_arhiiv[[c for c in teatis_arhiiv.columns if c != "geometry"]]],
    ignore_index=True
)

# Aasta veerg YoY analüüsi jaoks
teatis_all["aasta"] = pd.to_datetime(
    teatis_all["otsus_kinnitatud_kp"], errors="coerce").dt.year

# Raieliigi eestikeelne nimetus
teatis_all["raieliik"] = teatis_all["too_kood"].map(TOO_NIMED).fillna(teatis_all["too_kood"])

# Viimane teatis iga eraldise kohta (liitmiseks põhitabeliga)
teatis_latest = (
    teatis_all
    .sort_values("otsus_kinnitatud_kp", ascending=False)
    .groupby(["katastri_nr", "eraldise_nr"], dropna=False)
    .first()
    .reset_index()[
        ["katastri_nr", "eraldise_nr", "too_kood", "raieliik",
         "otsus", "otsus_kinnitatud_kp", "raiutav_maht", "pindala", "aasta"]
    ]
    .rename(columns={
        "pindala": "teatis_pindala",
        "aasta": "teatis_aasta",
        "otsus_kinnitatud_kp": "teatis_kp",
    })
)

print(f"\n✓ Teatised laetud: {len(teatis_all)} kokku ({len(teatis)} aktiivset + {len(teatis_arhiiv)} arhiivis)")
print(f"  Otsused: {teatis_all['otsus'].value_counts().to_dict()}")
print(f"  Raieliigid:\n{teatis_all['raieliik'].value_counts().to_string()}")

  Laadin: metsaregister:teatis... 10000 rida
  Laadin: metsaregister:teatis_arhiiv... 10000 rida

✓ Teatised laetud: 20000 kokku (10000 aktiivset + 10000 arhiivis)
  Otsused: {'JAH': 20000}
  Raieliigid:
raieliik
Lageraie         10420
Harvendusraie     5240
Sanitaarraie      1576
KR                 856
RD                 801
AR                 738
Turberaie          257
VE                  65
Valikraie           39
HL                   8


In [6]:
# ── Lahter 5: Liida eraldised ja teatised (tabeliliitmine) ────────────────────
merged = eraldis.merge(
    teatis_latest,
    on=["katastri_nr", "eraldise_nr"],
    how="left"
)
merged["on_teatis"] = merged["too_kood"].notna()

# Omandivormi eestikeelne nimetus
OMAND_NIMED = {
    "R": "Riigi", "T": "Riigi (RMK)", "F": "Eraomand",
    "J": "Juriidiline isik", "Y": "Muu eraomand",
    "X": "Teadmata", "E": "Eraomand", "M": "Munitsipaal", "A": "Muu"
}
merged["omanditvorm"] = merged["omandivorm_kood"].map(OMAND_NIMED).fillna(merged["omandivorm_kood"])

print(f"✓ Liitmine valmis: {len(merged)} eraldist")
print(f"\n  Teatisega / ilma:")
print(merged["on_teatis"].value_counts().rename({True: "On teatis", False: "Ei ole teatist"}).to_string())
print(f"\n  Omanditvorm:")
print(merged.groupby("omanditvorm")["pindala"].sum().sort_values(ascending=False).apply(lambda x: f"{x:.0f} ha").to_string())

✓ Liitmine valmis: 10000 eraldist

  Teatisega / ilma:
on_teatis
Ei ole teatist    7830
On teatis         2170

  Omanditvorm:
omanditvorm
Riigi               9314 ha
Eraomand            2575 ha
Juriidiline isik    1815 ha
Muu eraomand          23 ha
Munitsipaal           15 ha
Muu                    2 ha


In [7]:
# ── Lahter 6: Lae kaitsealad ja liida ruumiliselt ────────────────────────────
# Kaitsealad pole seotud katastri_nr-ga → kasutame punktis-polügoon analüüsi
# (eraldise tsentroid → kas asub kaitsealal?)

kaitsealad  = fetch_wfs(WFS_EELIS, "eelis:kr_kaitseala",  bbox=BBOX)
natura_hab  = fetch_wfs(WFS_EELIS, "eelis:kr_loodusala",  bbox=BBOX)
natura_bird = fetch_wfs(WFS_EELIS, "eelis:kr_linnuala",   bbox=BBOX)
vep         = fetch_wfs(WFS_EELIS, "eelis:kr_vep",        bbox=BBOX)

# Tsentroidid ruumiliseks liitmiseks
pts = merged.copy()
pts = pts.set_geometry(pts.geometry.centroid)

# 1) Kaitsealad (KMKA, KLKA, KRP jne)
if kaitsealad is not None and len(kaitsealad):
    ka = kaitsealad[["nimi","tyyp","geometry"]].rename(
        columns={"nimi":"kaitseala_nimi","tyyp":"kaitseala_tyyp"})
    pts = pts.sjoin(ka, how="left", predicate="within")
    pts["kaitseala_tyyp_nimi"] = pts["kaitseala_tyyp"].map(KAITSEALA_NIMED).fillna(pts.get("kaitseala_tyyp",""))
    pts.drop(columns=[c for c in ["index_right"] if c in pts.columns], inplace=True)

# 2) Natura 2000 (elupaiga- ja linnualad)
if natura_hab is not None or natura_bird is not None:
    natura_parts = [df[["nimi","geometry"]] for df in [natura_hab, natura_bird] if df is not None and len(df)]
    if natura_parts:
        natura = pd.concat(natura_parts).rename(columns={"nimi":"natura_nimi"})
        pts = pts.sjoin(natura, how="left", predicate="within")
        pts.drop(columns=[c for c in ["index_right"] if c in pts.columns], inplace=True)

# 3) Vääriselupaigad (VEP)
if vep is not None and len(vep):
    v = vep[["nimi","veptyyp","staatus","geometry"]].rename(
        columns={"nimi":"vep_nimi","veptyyp":"vep_tyyp","staatus":"vep_staatus"})
    pts = pts.sjoin(v, how="left", predicate="within")
    pts.drop(columns=[c for c in ["index_right"] if c in pts.columns], inplace=True)

# Lipud
pts["on_kaitseala"]  = pts.get("kaitseala_nimi",  pd.Series(dtype=str)).notna()
pts["on_natura2000"] = pts.get("natura_nimi",      pd.Series(dtype=str)).notna()
pts["on_vep"]        = pts.get("vep_nimi",         pd.Series(dtype=str)).notna()

print(f"\n✓ Kaitsealad liidetud")
print(f"  Kaitsealal: {pts['on_kaitseala'].sum()} eraldist")
print(f"  Natura 2000: {pts['on_natura2000'].sum()} eraldist")
print(f"  VEP: {pts['on_vep'].sum()} eraldist")

  Laadin: eelis:kr_kaitseala... 42 rida
  Laadin: eelis:kr_loodusala... 15 rida
  Laadin: eelis:kr_linnuala... 3 rida
  Laadin: eelis:kr_vep... 506 rida

✓ Kaitsealad liidetud
  Kaitsealal: 1732 eraldist
  Natura 2000: 2846 eraldist
  VEP: 136 eraldist


In [8]:
# ── Lahter 7: Aasta-üle-aasta (YoY) kokkuvõte ────────────────────────────────
yoy = (
    teatis_all[teatis_all["otsus"] == "JAH"]
    .groupby(["aasta", "too_kood", "raieliik"], dropna=False)
    .agg(
        teatiste_arv  = ("katastri_nr", "count"),
        kogupindala_ha = ("pindala",    "sum"),
        kogumaht_m3   = ("raiutav_maht", "sum"),
    )
    .reset_index()
    .sort_values(["aasta", "kogupindala_ha"], ascending=[True, False])
)

print("✓ YoY tabel valmis")
print(f"\n  Aastad kaetud: {int(yoy['aasta'].min())} – {int(yoy['aasta'].max())}")
print("\n  Viimase 5 aasta kokkuvõte:")
recent = yoy[yoy["aasta"] >= yoy["aasta"].max() - 4]
print(recent.groupby("raieliik")[["kogupindala_ha","kogumaht_m3"]].sum()
      .sort_values("kogupindala_ha", ascending=False)
      .applymap(lambda x: f"{x:.0f}").to_string())

✓ YoY tabel valmis

  Aastad kaetud: 2016 – 2026

  Viimase 5 aasta kokkuvõte:
              kogupindala_ha kogumaht_m3
raieliik                                
Lageraie                5283      818449
Harvendusraie           3724      149413
Sanitaarraie            1510       15906
AR                       407       23432
KR                       191        5967
RD                       132        9729
Turberaie                 57        4478
Valikraie                 54        4416
VE                        25        1569
HL                         3          50


/tmp/ipykernel_1588/2388431700.py:20: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  .applymap(lambda x: f"{x:.0f}").to_string())


In [9]:
# ── Lahter 8: Ekspordi CSV failid ─────────────────────────────────────────────
# Eemalda geomeetria veerg (Sheets ei vaja seda)
mets_out = pts.drop(columns=["geometry"], errors="ignore")

# Ümarda arvud loetavamaks
for col in ["pindala", "tagavara_1_ha", "juurdekasv", "teatis_pindala"]:
    if col in mets_out.columns:
        mets_out[col] = mets_out[col].round(2)

mets_out.to_csv("laanema_mets.csv",  index=False, encoding="utf-8-sig")
yoy.to_csv(     "laanema_yoy.csv",   index=False, encoding="utf-8-sig")

print("✓ Failid salvestatud!")
print(f"  laanema_mets.csv — {len(mets_out)} rida, {len(mets_out.columns)} veergu")
print(f"  laanema_yoy.csv  — {len(yoy)} rida")
print("")
print("Laadi failid alla: vasakul paneelil Files (📁) → parem klikk failil → Download")
print("Seejärel impordi Google Sheets'i: File → Import → Upload")

# Veerunimed ülevaateks
print(f"\n  Põhitabeli veerud:")
for i, col in enumerate(mets_out.columns, 1):
    print(f"    {i:2}. {col}")

✓ Failid salvestatud!
  laanema_mets.csv — 11150 rida, 32 veergu
  laanema_yoy.csv  — 86 rida

Laadi failid alla: vasakul paneelil Files (📁) → parem klikk failil → Download
Seejärel impordi Google Sheets'i: File → Import → Upload

  Põhitabeli veerud:
     1. katastri_nr
     2. eraldise_nr
     3. pindala
     4. peapuuliik_kood
     5. keskm_vanus
     6. omandivorm_kood
     7. kasvukoht_kood
     8. arengukl_kood
     9. tagavara_1_ha
    10. taius_1
    11. korgus
    12. juurdekasv
    13. mkood
    14. too_kood
    15. raieliik
    16. otsus
    17. teatis_kp
    18. raiutav_maht
    19. teatis_pindala
    20. teatis_aasta
    21. on_teatis
    22. omanditvorm
    23. kaitseala_nimi
    24. kaitseala_tyyp
    25. kaitseala_tyyp_nimi
    26. natura_nimi
    27. vep_nimi
    28. vep_tyyp
    29. vep_staatus
    30. on_kaitseala
    31. on_natura2000
    32. on_vep


## Mida saad Google Sheets'is teha

**`laanema_mets.csv` — põhitabel (üks eraldis = üks rida)**

| Küsimus | Filter / Pivot |
|---|---|
| Kui palju on puutumata metsa? | `on_teatis = FALSE` → summa `pindala` |
| Mis raie tüüpe rakendatakse enim? | Pivot: `raieliik` × `teatis_pindala` |
| Kas kaitsealadel raiutakse? | Filter `on_kaitseala = TRUE` + `on_teatis = TRUE` |
| Millist metsa kõige rohkem raiutakse? | Pivot: `peapuuliik_kood` × `teatis_pindala` |
| Eraomand vs riigimets? | Pivot: `omanditvorm` × `pindala` ja `on_teatis` |

**`laanema_yoy.csv` — aastane kokkuvõte**

| Küsimus | Filter / Pivot |
|---|---|
| Raie trend aastate lõikes? | Joonis: `aasta` × `kogupindala_ha` |
| Lageraie vs harvendusraie trend? | Pivot: `aasta` × `raieliik` × `kogupindala_ha` |
| Maht vs pindala? | Scatter: `kogupindala_ha` vs `kogumaht_m3` |

---
*Skaala suurendamiseks: muuda `BBOX` väärtust lahter 1-s. Kogu Eesti jaoks jäta BBOX välja (kuid siis võib alla laadimine kesta 5–10 minutit).*